# SAM2.1 + UNet for TWSI Segmentation
---

This notebook fine-tunes SAM2.1 with a custom UNet decoder for tactile walking surface
indicator segmentation. The SAM2.1 backbone (hiera_base_plus) is frozen, and a lightweight
U-Net decoder is trained on top of the 64×64 encoder features.

**Architecture:** Frozen SAM2.1 encoder → Custom UNet decoder (4 stages: 64→128→256→512→1024)
with skip connections for dense, class-specific segmentation.

Based on the [Roboflow SAM2.1 fine-tuning guide](https://blog.roboflow.com/fine-tune-sam-2-1/).

**Paper Reference:** GuideTWSI (ICRA 2026), Section VI-A

## 1. Installation

Clone the SAM2 repository and install dependencies.

In [ ]:
!git clone https://github.com/facebookresearch/sam2.git

In [ ]:
%cd sam2
!pip install -e .[dev] -q
!pip install supervision -q

Download pre-trained checkpoints:

In [ ]:
!cd checkpoints && ./download_ckpts.sh

## 2. Configuration

Load training hyperparameters from the config file.

In [ ]:
import yaml

with open("../configs/sam2_unet.yaml", "r") as f:
    config = yaml.safe_load(f)

EPOCHS = config["training"]["epochs"]  # 100
BATCH_SIZE = config["training"]["batch_size"]  # 4
LEARNING_RATE = config["training"]["learning_rate"]  # 0.0001
SAM2_CHECKPOINT = config["training"]["sam2_checkpoint"]
SAM2_CONFIG = config["training"]["sam2_config"]

print(f"Training config: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}")

## 3. Download Dataset

Download the GuideTWSI dataset from HuggingFace Hub.

**Data format:** Flat directories with paired `.jpg` images and `.json` files containing
RLE segmentation annotations.

In [ ]:
# Download dataset from HuggingFace
# !huggingface-cli download guidedogrobot-tactile/GuideTWSI --local-dir ./data

# Set data paths
TRAIN_DIR = "data/train"  # Contains image.jpg + image.json pairs
VAL_DIR = "data/val"
TEST_DIR = "data/test"

## 4. Training

**Important notes before training:**
- If installation errors persist, update `pyproject.toml`: change `"setuptools>=61.0"` to `"setuptools>=62.3.0,<75.9"`
- Update `train.yaml` with your dataset path, ground truth path, and preferred checkpoint
- Adjust batch size based on available GPU memory

In [ ]:
# Download the optimized training YAML configuration
# Update the train.yaml with:
#   - Dataset location path
#   - Ground truth location path  
#   - Checkpoint file (sam2.1_hiera_base_plus.pt)
#   - Batch size, workers, epochs as needed

!python training/train.py -c 'configs/train.yaml' --use-cluster 0 --num-gpus 1

## 5. Inference

Load the fine-tuned model and run inference on test images.

In [ ]:
import torch
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import supervision as sv
import os
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Enable bfloat16 for efficiency
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Load fine-tuned model
checkpoint = ""  # Path to your fine-tuned checkpoint
model_cfg = SAM2_CONFIG
sam2 = build_sam2(model_cfg, checkpoint, device="cuda")
mask_generator = SAM2AutomaticMaskGenerator(sam2)

# Load base model for comparison
checkpoint_base = f"checkpoints/{SAM2_CHECKPOINT}"
sam2_base = build_sam2(model_cfg, checkpoint_base, device="cuda")
mask_generator_base = SAM2AutomaticMaskGenerator(sam2_base)

print("Models loaded successfully.")

In [ ]:
# Run inference on a random validation image
validation_set = [f for f in os.listdir(VAL_DIR) if f.endswith(".jpg")]
image_name = random.choice(validation_set)
image_path = os.path.join(VAL_DIR, image_name)
opened_image = np.array(Image.open(image_path).convert("RGB"))

# Fine-tuned model predictions
result = mask_generator.generate(opened_image)
detections = sv.Detections.from_sam(sam_result=result)
mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)
annotated_image = mask_annotator.annotate(opened_image.copy(), detections=detections)

# Base model predictions
base_result = mask_generator_base.generate(opened_image)
base_detections = sv.Detections.from_sam(sam_result=base_result)
base_annotated = mask_annotator.annotate(opened_image.copy(), detections=base_detections)

# Visualize comparison
sv.plot_images_grid(
    images=[annotated_image, base_annotated],
    titles=["Fine-Tuned SAM2.1", "Base SAM2.1"],
    grid_size=(1, 2),
)

## 6. Evaluation

Evaluate the fine-tuned model on the full test set with binary segmentation metrics.

In [ ]:
import sys
from tqdm import tqdm

sys.path.insert(0, "..")
from evaluation.metrics import compute_binary_metrics, aggregate_metrics

test_images = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(".jpg")])
all_metrics = []

for image_name in tqdm(test_images, desc="Evaluating"):
    image_path = os.path.join(TEST_DIR, image_name)
    mask_path = os.path.join(TEST_DIR, "masks", image_name.replace(".jpg", ".png"))
    if not os.path.exists(mask_path):
        mask_path = os.path.join(TEST_DIR, "masks", image_name.replace(".jpg", "_mask.jpg"))
    if not os.path.exists(mask_path):
        continue

    opened_image = np.array(Image.open(image_path).convert("RGB"))
    ground_truth = np.array(Image.open(mask_path).convert("L"))

    result = mask_generator.generate(opened_image)

    # Combine all predicted masks
    pred_mask = np.zeros(ground_truth.shape, dtype=bool)
    for m in result:
        seg = np.array(m["segmentation"])
        if seg.shape == pred_mask.shape:
            pred_mask |= seg

    pred_binary = (pred_mask.astype(np.uint8) * 255)
    metrics = compute_binary_metrics(ground_truth, pred_binary)
    all_metrics.append(metrics)

avg = aggregate_metrics(all_metrics)
print(f"\nResults over {len(all_metrics)} test images:")
for k, v in avg.items():
    print(f"  {k}: {v:.4f}")